# Fine-Tuning a Random Forest Classifier for Azure ML

This notebook demonstrates how to train and fine-tune (hyperparameter optimise)
a `sklearn` Random Forest classifier inside an **Azure Machine Learning** environment,
using **PyTorch** as the data-wrangling and preprocessing backbone.

## Workflow

| Step | What happens |
|------|--------------|
| 1 | Connect to an Azure ML Workspace |
| 2 | Load the Wine dataset; preprocess with PyTorch tensors |
| 3 | Train a baseline Random Forest |
| 4 | Fine-tune hyperparameters with `RandomizedSearchCV` |
| 5 | Log all metrics to an Azure ML Experiment Run |
| 6 | Register the best model in the Azure ML Model Registry |

### Prerequisites

```bash
pip install azureml-sdk torch scikit-learn scipy joblib
```

An Azure ML workspace is required. Set the four variables in the **Workspace Setup**
cell below, or drop a `config.json` (downloaded from the Azure portal) next to this
notebook and call `Workspace.from_config()` instead.

> **Note:** Random Forests are not gradient-based models — 'fine-tuning' here means
> systematic hyperparameter search, the standard optimisation technique for
> ensemble tree methods.

## 1  Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
import json
import os
import time

# ── Numeric / ML ──────────────────────────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform
import joblib

# ── Azure ML ──────────────────────────────────────────────────────────────────
from azureml.core import Workspace, Experiment, Run, Model
from azureml.core.authentication import InteractiveLoginAuthentication

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
print(f'PyTorch  {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2  Azure ML Workspace Setup

Replace the placeholder strings with your actual Azure subscription details,
or use `Workspace.from_config()` if `config.json` is present.

In [ ]:
# ── Workspace credentials — edit these or use config.json ─────────────────────
SUBSCRIPTION_ID = os.environ.get('AZURE_SUBSCRIPTION_ID', '<your-subscription-id>')
RESOURCE_GROUP  = os.environ.get('AZURE_RESOURCE_GROUP',  '<your-resource-group>')
WORKSPACE_NAME  = os.environ.get('AZURE_ML_WORKSPACE',    '<your-workspace-name>')
EXPERIMENT_NAME = 'rf-wine-finetune'

# ── Connect ───────────────────────────────────────────────────────────────────
try:
    # Preferred: read config.json dropped alongside this notebook
    ws = Workspace.from_config()
    print(f'Loaded workspace from config.json: {ws.name}')
except Exception:
    # Fallback: construct from explicit credentials
    ws = Workspace(
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        workspace_name=WORKSPACE_NAME,
    )
    print(f'Connected to workspace: {ws.name} (region: {ws.location})')

experiment = Experiment(workspace=ws, name=EXPERIMENT_NAME)
print(f'Experiment: {experiment.name}')

## 3  Data Loading and PyTorch Preprocessing

We use scikit-learn's built-in **Wine** dataset (178 samples, 13 features,
3 classes) as a small but realistic multiclass benchmark.

PyTorch `TensorDataset` + `DataLoader` provide:
- GPU-compatible tensor conversion (ready for any downstream deep model)
- Reproducible shuffled mini-batch iteration
- A clean hand-off point if you later swap the RF for a neural network

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
wine = load_wine()
X_raw, y_raw = wine.data, wine.target
feature_names = wine.feature_names
class_names   = wine.target_names

print(f'Dataset shape : {X_raw.shape}')
print(f'Classes       : {class_names}')
print(f'Class balance : {np.bincount(y_raw)}')

# ── Train / test split (stratified) ──────────────────────────────────────────
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# ── Standardise (fit on train only) ──────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_np)
X_test_scaled  = scaler.transform(X_test_np)

# ── Convert to PyTorch tensors ────────────────────────────────────────────────
def to_tensor(X, y):
    """Return float32 feature tensor and int64 label tensor on DEVICE."""
    return (
        torch.tensor(X, dtype=torch.float32).to(DEVICE),
        torch.tensor(y, dtype=torch.int64).to(DEVICE),
    )

X_train_t, y_train_t = to_tensor(X_train_scaled, y_train_np)
X_test_t,  y_test_t  = to_tensor(X_test_scaled,  y_test_np)

# ── DataLoaders (useful for batched inference or downstream neural models) ─────
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0,
)
test_loader  = DataLoader(
    TensorDataset(X_test_t,  y_test_t),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)

print(f'Train batches : {len(train_loader)} ({len(train_loader.dataset)} samples)')
print(f'Test  batches : {len(test_loader)}  ({len(test_loader.dataset)}  samples)')

### 3a  PyTorch Feature Statistics

We inspect per-feature mean and standard deviation using tensor operations
to confirm the scaler worked correctly.

In [ ]:
# Compute statistics via PyTorch (mean ≈ 0, std ≈ 1 after scaling)
feat_mean = X_train_t.mean(dim=0).cpu().numpy()
feat_std  = X_train_t.std(dim=0).cpu().numpy()

print(f'{"Feature":<30} {"Mean":>8} {"Std":>8}')
print('-' * 50)
for name, m, s in zip(feature_names, feat_mean, feat_std):
    print(f'{name:<30} {m:8.4f} {s:8.4f}')

## 4  Baseline Random Forest

We start an Azure ML Run immediately so all metrics — baseline and fine-tuned —
are captured in the same experiment.

In [ ]:
# ── Start Azure ML run ────────────────────────────────────────────────────────
run = experiment.start_logging(snapshot_directory=None)
print(f'Run ID: {run.id}')

# ── Baseline hyperparameters ──────────────────────────────────────────────────
BASELINE_PARAMS = {
    'n_estimators' : 100,
    'max_depth'    : None,   # fully grown
    'min_samples_split': 2,
    'min_samples_leaf' : 1,
    'max_features' : 'sqrt',
    'random_state' : 42,
    'n_jobs'       : -1,
}

# Log baseline params so they appear on the Azure ML run page
for k, v in BASELINE_PARAMS.items():
    run.log(f'baseline_{k}', v if v is not None else 'None')

# ── Train ─────────────────────────────────────────────────────────────────────
# sklearn expects plain NumPy arrays; pull back from GPU tensors
X_tr = X_train_t.cpu().numpy()
y_tr = y_train_t.cpu().numpy()
X_te = X_test_t.cpu().numpy()
y_te = y_test_t.cpu().numpy()

t0 = time.perf_counter()
baseline_rf = RandomForestClassifier(**BASELINE_PARAMS)
baseline_rf.fit(X_tr, y_tr)
baseline_train_time = time.perf_counter() - t0

# ── Evaluate ──────────────────────────────────────────────────────────────────
baseline_preds = baseline_rf.predict(X_te)
baseline_acc   = accuracy_score(y_te, baseline_preds)
baseline_f1    = f1_score(y_te, baseline_preds, average='weighted')

run.log('baseline_accuracy',    round(baseline_acc, 4))
run.log('baseline_f1_weighted', round(baseline_f1, 4))
run.log('baseline_train_time_s', round(baseline_train_time, 3))

print(f'Baseline accuracy : {baseline_acc:.4f}')
print(f'Baseline F1       : {baseline_f1:.4f}')
print(f'Train time        : {baseline_train_time:.3f}s')

## 5  Hyperparameter Fine-Tuning

We use **`RandomizedSearchCV`** with a stratified 5-fold cross-validation scheme.
This is the standard approach for fine-tuning tree ensemble hyperparameters:

| Hyperparameter | Search range | Effect |
|---|---|---|
| `n_estimators` | 50–500 | More trees → lower variance, higher cost |
| `max_depth` | 3–20, None | Controls overfitting |
| `min_samples_split` | 2–20 | Minimum samples to split a node |
| `min_samples_leaf` | 1–10 | Minimum samples at each leaf |
| `max_features` | sqrt, log2, 0.3–0.9 | Feature subset per split |
| `bootstrap` | True, False | Sample with/without replacement |

In [ ]:
# ── Search space ──────────────────────────────────────────────────────────────
param_dist = {
    'n_estimators'      : randint(50, 501),
    'max_depth'         : [None] + list(range(3, 21)),
    'min_samples_split' : randint(2, 21),
    'min_samples_leaf'  : randint(1, 11),
    'max_features'      : ['sqrt', 'log2', 0.3, 0.5, 0.7, 0.9],
    'bootstrap'         : [True, False],
}

N_ITER  = 50   # number of random configurations to try
CV_FOLDS = 5

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=N_ITER,
    cv=cv,
    scoring='f1_weighted',
    refit=True,        # refit the best estimator on the full training set
    verbose=1,
    random_state=42,
    n_jobs=-1,
)

print(f'Running RandomizedSearchCV: {N_ITER} iterations × {CV_FOLDS}-fold CV ...')
t0 = time.perf_counter()
search.fit(X_tr, y_tr)
search_time = time.perf_counter() - t0
print(f'Search complete in {search_time:.1f}s')

In [ ]:
# ── Best configuration ────────────────────────────────────────────────────────
best_params = search.best_params_
best_cv_f1  = search.best_score_
best_rf     = search.best_estimator_

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:<22} = {v}')
print(f'\nBest CV F1 (weighted): {best_cv_f1:.4f}')

# ── Evaluate best model on held-out test set ──────────────────────────────────
best_preds = best_rf.predict(X_te)
best_acc   = accuracy_score(y_te, best_preds)
best_f1    = f1_score(y_te, best_preds, average='weighted')

print(f'\nTest accuracy : {best_acc:.4f}  (baseline: {baseline_acc:.4f})')
print(f'Test F1       : {best_f1:.4f}  (baseline: {baseline_f1:.4f})')

## 6  Log Results to Azure ML

In [ ]:
# ── Log best hyperparameters ──────────────────────────────────────────────────
for k, v in best_params.items():
    run.log(f'best_{k}', v if v is not None else 'None')

# ── Log performance metrics ───────────────────────────────────────────────────
run.log('best_cv_f1_weighted',  round(best_cv_f1, 4))
run.log('best_test_accuracy',   round(best_acc,   4))
run.log('best_test_f1_weighted',round(best_f1,    4))
run.log('search_time_s',        round(search_time, 1))
run.log('improvement_accuracy', round(best_acc - baseline_acc, 4))
run.log('improvement_f1',       round(best_f1  - baseline_f1,  4))
run.log('n_search_iterations',  N_ITER)
run.log('cv_folds',             CV_FOLDS)

# ── Log per-class F1 ──────────────────────────────────────────────────────────
per_class_f1 = f1_score(y_te, best_preds, average=None)
for cls, f1 in zip(class_names, per_class_f1):
    run.log(f'f1_{cls}', round(float(f1), 4))

print('Metrics logged to Azure ML run.')
print(classification_report(y_te, best_preds, target_names=class_names))

## 7  Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_te, best_preds)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[0],
)
axes[0].set_title('Confusion Matrix — Best RF')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ── Feature importances ───────────────────────────────────────────────────────
importances = best_rf.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]
axes[1].barh(
    range(len(sorted_idx)),
    importances[sorted_idx],
    color='steelblue',
)
axes[1].set_yticks(range(len(sorted_idx)))
axes[1].set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=9)
axes[1].invert_yaxis()
axes[1].set_title('Feature Importances — Best RF')
axes[1].set_xlabel('Mean decrease in impurity')

plt.tight_layout()
plot_path = 'rf_results.png'
plt.savefig(plot_path, dpi=120)
run.log_image('rf_results', path=plot_path)   # upload to Azure ML run
print(f'Plot saved and uploaded: {plot_path}')
plt.show()

## 8  PyTorch Batch Inference Demo

We loop through the test `DataLoader` (one batch at a time) to produce predictions,
demonstrating how the PyTorch data pipeline feeds into the RF predictor.

In [ ]:
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        # Move to CPU numpy for sklearn
        batch_np = batch_X.cpu().numpy()
        preds    = best_rf.predict(batch_np)
        all_preds.extend(preds.tolist())
        all_labels.extend(batch_y.cpu().numpy().tolist())

# Verify identical to earlier evaluation
batch_acc = accuracy_score(all_labels, all_preds)
print(f'Batch-inference accuracy: {batch_acc:.4f}  (should match {best_acc:.4f})')
assert np.isclose(batch_acc, best_acc), 'Mismatch — check DataLoader ordering'
print('Assertion passed — batch inference results are consistent.')

## 9  Save and Register Model in Azure ML Model Registry

In [ ]:
MODEL_DIR  = 'outputs'
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Persist artefacts ─────────────────────────────────────────────────────────
model_path  = os.path.join(MODEL_DIR, 'rf_wine_best.pkl')
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')
meta_path   = os.path.join(MODEL_DIR, 'metadata.json')

joblib.dump(best_rf, model_path)
joblib.dump(scaler,  scaler_path)

metadata = {
    'experiment'   : EXPERIMENT_NAME,
    'run_id'       : run.id,
    'dataset'      : 'sklearn wine',
    'n_features'   : int(X_raw.shape[1]),
    'n_classes'    : len(class_names),
    'class_names'  : list(class_names),
    'feature_names': list(feature_names),
    'best_params'  : {k: (str(v) if v is None else v) for k, v in best_params.items()},
    'test_accuracy': round(best_acc, 4),
    'test_f1'      : round(best_f1,  4),
    'torch_version': torch.__version__,
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved: {model_path}')
print(f'Saved: {scaler_path}')
print(f'Saved: {meta_path}')

# ── Upload outputs folder to Azure ML run ─────────────────────────────────────
run.upload_folder(name='outputs', path=MODEL_DIR)
print('Artefacts uploaded to Azure ML run.')

In [ ]:
# ── Register model in the workspace Model Registry ───────────────────────────
registered_model = run.register_model(
    model_name='rf-wine-classifier',
    model_path='outputs/rf_wine_best.pkl',
    tags={
        'framework'    : 'sklearn',
        'preprocessing': 'pytorch-tensor-pipeline',
        'dataset'      : 'wine',
        'test_accuracy': str(round(best_acc, 4)),
        'test_f1'      : str(round(best_f1,  4)),
    },
    description=(
        f'Random Forest classifier on Wine dataset. '
        f'Fine-tuned via RandomizedSearchCV ({N_ITER} iterations, {CV_FOLDS}-fold CV). '
        f'Test accuracy {best_acc:.4f}, F1 {best_f1:.4f}.'
    ),
)
print(f'Registered: {registered_model.name}  version {registered_model.version}')

## 10  Complete the Azure ML Run

In [ ]:
run.complete()
print(f'Run {run.id} completed.')
print(f'View at: {run.get_portal_url()}')

## 11  Summary

| | Baseline RF | Fine-Tuned RF |
|---|---|---|
| `n_estimators` | 100 | (search result) |
| `max_depth` | None | (search result) |
| Test Accuracy | — | — |
| Test F1 (weighted) | — | — |

> Run the cells above to populate this table with your actual results.

### What was logged to Azure ML

- All baseline and best hyperparameters
- Test accuracy + F1 for baseline and fine-tuned model
- Per-class F1 scores
- Confusion matrix + feature importance plot (as image)
- Model artefacts (`rf_wine_best.pkl`, `scaler.pkl`, `metadata.json`)
- Model registered in the workspace registry as `rf-wine-classifier`

### Extending this notebook

- **Replace the dataset** — point `X_raw, y_raw` at your own Azure Dataset
- **Use HyperDrive** — swap `RandomizedSearchCV` for `azureml.train.hyperdrive`
  to parallelise the search across compute clusters
- **Add a neural baseline** — the `DataLoader` wiring is already in place;
  define a `torch.nn.Module` classifier and compare against the RF
- **Deploy** — use `Model.deploy()` with an `InferenceConfig` to serve the
  registered model as a REST endpoint on ACI or AKS